In [1]:
import requests
import polars as pl
import os
import json

# ==============================================================================
# MILESTONE 2.1e: FULL BOUNDARY ENUMERATION + VOLUME SEARCH (BULLETPROOF)
# Goal: (a) Confirm SSEN-S exists, (b) Find volume data, (c) Lock data contract
# ==============================================================================

API_BASE = "https://api.neso.energy/api/3/action"
TIMEOUT_SECONDS = 10  # FAIL LOUDLY: No silent hangs

# Datasets to fully enumerate
ENUMERATION_TARGETS = [
    {
        "name": "Thermal Constraint Costs 23-24",
        "resource_id": "75c9c564-af38-4421-a461-a612a6921212",
        "package_id": "f0055054-c55c-4068-a01c-61da4334e58f",
    },
    {
        "name": "Day Ahead Constraint Flows and Limits",
        "resource_id": "38a18ec1-9e40-465d-93fb-301e80fd1352",
        "package_id": "cf3cbc92-2d5d-4c2b-bd29-e11a21070b26",
    },
]

# What we are looking for (case-insensitive matching)
TARGET_IDENTIFIERS = ["SCOTEX", "SSEN-S", "SSEN S", "SSE-N", "B6", "B2", "SSEN", "SSHARN"]


def get_all_unique_constraint_groups(resource_id: str, name: str) -> list:
    """
    Fetches a safe sample (2000 rows) to extract all unique Constraint Groups.
    2000 rows is more than enough to capture all ~15 unique GB constraint groups.
    """
    print(f"\n  Querying unique 'Constraint Group' values for: {name}")

    try:
        params = {"resource_id": resource_id, "limit": 2000, "offset": 0}
        resp = requests.get(f"{API_BASE}/datastore_search", params=params, timeout=TIMEOUT_SECONDS)
        resp.raise_for_status()
        data = resp.json()

        if not data.get("success"):
            raise RuntimeError(f"API error: {data.get('error')}")

        records = data["result"]["records"]
        total_available = data["result"].get("total", len(records))
        print(f"  ℹ️  Fetched {len(records)} of {total_available} total rows")

        df = pl.DataFrame(records)

        if "Constraint Group" in df.columns:
            groups = df["Constraint Group"].unique().drop_nulls().sort().to_list()
            return groups
        else:
            print(f"  ⚠️ No 'Constraint Group' column. Available: {df.columns}")
            return []

    except requests.exceptions.Timeout:
        print(f"  ❌ TIMEOUT: Request exceeded {TIMEOUT_SECONDS} seconds. Failing loudly.")
        return []
    except Exception as e:
        print(f"  ❌ Failed: {e}")
        return []


def inspect_full_schema(resource_id: str, name: str) -> pl.DataFrame | None:
    """Fetch 3 rows to confirm full column list."""
    try:
        params = {"resource_id": resource_id, "limit": 3}
        resp = requests.get(f"{API_BASE}/datastore_search", params=params, timeout=TIMEOUT_SECONDS)
        resp.raise_for_status()
        data = resp.json()
        if data.get("success") and data["result"]["records"]:
            df = pl.DataFrame(data["result"]["records"])
            print(f"  📋 Full schema: {df.columns}")
            return df
    except requests.exceptions.Timeout:
        print(f"  ❌ TIMEOUT: Schema fetch exceeded {TIMEOUT_SECONDS} seconds.")
    except Exception as e:
        print(f"  ❌ Schema fetch failed: {e}")
    return None


def check_package_for_volume_resources(package_id: str, package_name: str) -> None:
    """Check if the parent package has sibling resources with volume data."""
    print(f"\n  🔍 Checking parent package '{package_name}' for volume resources...")
    try:
        resp = requests.get(
            f"{API_BASE}/package_show",
            params={"id": package_id},
            timeout=TIMEOUT_SECONDS
        )
        resp.raise_for_status()
        data = resp.json()

        if data.get("success"):
            resources = data["result"].get("resources", [])
            volume_found = False
            for res in resources:
                res_name = res.get("name", "").lower()
                res_desc = res.get("description", "").lower()
                if any(kw in res_name or kw in res_desc for kw in ["volume", "mwh", "energy"]):
                    print(f"    🎯 VOLUME RESOURCE FOUND: {res['name']}")
                    print(f"       ID: {res['id']}")
                    print(f"       Format: {res.get('format', 'unknown')}")
                    volume_found = True
            
            if not volume_found:
                print("    ⚠️ No obvious volume/MWh resource found in this package.")
                
            print(f"    All resources in package ({len(resources)}):")
            for res in resources:
                print(f"      - {res['name']} ({res.get('format', '?')}) [ID: {res['id'][:8]}...]")
    except requests.exceptions.Timeout:
        print(f"  ❌ TIMEOUT: Package lookup exceeded {TIMEOUT_SECONDS} seconds.")
    except Exception as e:
        print(f"  ❌ Package lookup failed: {e}")


def match_targets(groups: list) -> dict:
    """Check which of our target boundaries appear in the group list."""
    results = {}
    groups_upper = [str(g).upper() for g in groups]
    for target in TARGET_IDENTIFIERS:
        results[target] = any(target.upper() in g for g in groups_upper)
    return results


if __name__ == "__main__":
    os.makedirs("data/intermediate", exist_ok=True)

    print("=" * 80)
    print("MILESTONE 2.1e: FULL BOUNDARY ENUMERATION (TIMEOUT-ENFORCED)")
    print("=" * 80)

    all_findings = {}

    for target in ENUMERATION_TARGETS:
        print(f"\n{'─' * 80}")
        print(f"DATASET: {target['name']}")
        print(f"Resource: {target['resource_id']}")
        print(f"{'─' * 80}")

        # Step 1: Get full schema
        sample_df = inspect_full_schema(target["resource_id"], target["name"])

        # Step 2: Enumerate ALL constraint groups
        groups = get_all_unique_constraint_groups(target["resource_id"], target["name"])

        if groups:
            print(f"\n  ✅ ALL Constraint Groups ({len(groups)}):")
            for g in groups:
                print(f"     • {g}")

            # Step 3: Match against targets
            matches = match_targets(groups)
            print(f"\n  🎯 TARGET MATCH RESULTS:")
            for identifier, found in matches.items():
                status = "✅ FOUND" if found else "❌ NOT FOUND"
                print(f"     {identifier}: {status}")

            all_findings[target["name"]] = {"groups": groups, "matches": matches}

        # Step 4: Check for volume data in parent package
        check_package_for_volume_resources(target["package_id"], target["name"])

    # Final verdict
    print(f"\n{'=' * 80}")
    print("DATA CONTRACT VERDICT")
    print(f"{'=' * 80}")

    scotex_found = any(f["matches"].get("SCOTEX", False) or f["matches"].get("B6", False) for f in all_findings.values())
    ssen_found = any(
        f["matches"].get("SSEN-S", False) or f["matches"].get("SSEN S", False) or 
        f["matches"].get("SSE-N", False) or f["matches"].get("B2", False) or f["matches"].get("SSHARN", False)
        for f in all_findings.values()
    )

    print(f"\n  SCOTEX (B6) confirmed in public data: {'✅ YES' if scotex_found else '❌ NO'}")
    print(f"  SSEN-S (B2) or alias confirmed in public data: {'✅ YES' if ssen_found else '❌ NO'}")

    if scotex_found and ssen_found:
        print("\n  ✅ DATA CONTRACT CAN BE LOCKED.")
        print("  → Proceed to Milestone 2.2 (Scottish Event Profiler).")
    elif scotex_found and not ssen_found:
        print("\n  ⚠️ PARTIAL CONTRACT: SCOTEX confirmed, SSEN-S absent or aliased.")
        print("  → ACTION: We may need to map 'SSHARN' or 'SSE-N' to the SSEN-S (B2) boundary.")
    else:
        print("\n  ❌ DATA CONTRACT CANNOT BE LOCKED.")
        print("  → Pivot to BOALF-to-Boundary mapping strategy.")

    # Save findings
    with open("data/intermediate/01e_boundary_enumeration_results.json", "w") as f:
        json.dump(all_findings, f, indent=2, default=str)
    print(f"\n  📁 Results saved to: data/intermediate/01e_boundary_enumeration_results.json")

MILESTONE 2.1e: FULL BOUNDARY ENUMERATION (TIMEOUT-ENFORCED)

────────────────────────────────────────────────────────────────────────────────
DATASET: Thermal Constraint Costs 23-24
Resource: 75c9c564-af38-4421-a461-a612a6921212
────────────────────────────────────────────────────────────────────────────────
  📋 Full schema: ['_id', 'Settlement Date', 'Constraint Group', 'Daily Cost (GBP)']

  Querying unique 'Constraint Group' values for: Thermal Constraint Costs 23-24
  ℹ️  Fetched 2000 of 2196 total rows

  ✅ ALL Constraint Groups (6):
     • ESTEX
     • SCOTEX
     • SEIMP
     • SSE-SP
     • SSHARN
     • SWALEX

  🎯 TARGET MATCH RESULTS:
     SCOTEX: ✅ FOUND
     SSEN-S: ❌ NOT FOUND
     SSEN S: ❌ NOT FOUND
     SSE-N: ❌ NOT FOUND
     B6: ❌ NOT FOUND
     B2: ❌ NOT FOUND
     SSEN: ❌ NOT FOUND
     SSHARN: ✅ FOUND

  🔍 Checking parent package 'Thermal Constraint Costs 23-24' for volume resources...
    ⚠️ No obvious volume/MWh resource found in this package.
    All resources